# Diffusion-ES: does a more diverse initial pool help?

**Hypothesis:** some failures happen because the diffusion model's 64 initial samples all sit in one mode (one lane, one speed profile), so ES has nothing safe to refine. A more diverse first pool would give ES a safe starting point and make the planner react better.

**Protocol** (e450 world model + `margin_tiebreak`, the best config so far):

1. **Diagnose:** on scenarios that fail in every ES run, re-plan at the decisions before the failure and check whether the *initial* pool contained any safe plan.
   * No safe plan in iteration 0 → diversity is the bottleneck, the hypothesis is plausible.
   * Safe plans existed but were not chosen → selection/scoring problem, more diversity will not help.
2. **Intervene:** swap only the initial-pool sampler (`initial_pool=`), same seed, same reward, same ES loop.
3. **Closed loop:** success on the candidate set, baseline vs variants.

Initial-pool samplers in `es_debug`: `standard_pool` (production), `oversample_fps(factor)` (draw factor×P, keep P by farthest-point sampling), `goal_mix(frac)` (part of the pool sampled with the goal masked). Write your own: `(planner, *, cond_bf, inst_cond_bf, inst_cond_mask_bf, features, pre_batch, rng) -> [1, P, T, D]`.

In [ ]:
import os
# Torch WM + JAX planner share one GPU; must be set before JAX is imported.
os.environ.setdefault("XLA_PYTHON_CLIENT_PREALLOCATE", "false")

import tensorflow as tf
tf.config.set_visible_devices([], "GPU")

import sys
from pathlib import Path

REPO_ROOT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
sys.path.insert(0, str(REPO_ROOT))
sys.path.insert(0, str(REPO_ROOT / "notebooks"))

%load_ext autoreload
%autoreload 2
%matplotlib inline

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from IPython.display import display

import es_debug as esd

## 1. Configuration

In [ ]:
SUBSET_DIR = "/zfsauton/scratch/sbellad/Diffusion_ES_sim/es_failure_subset_n1500_seed0"
TFRECORD_DIR = f"{SUBSET_DIR}/tfrecords/training"
LANE_GRAPH_DIR = "/zfsauton/scratch/mineuih/waymax_rs/lane_graphs"

# ---- models: best config so far (e450 WM + margin_tiebreak) -----------------
DIFFUSION_CKPT = "/zfsauton/scratch/sbellad/train_diffusion/diffusion_baseline_baseline_20260810_144559/epoch_0500"
WM_CKPT = "/zfsauton/scratch/ishanj/world_model/ckpts_multifuture_gaussian/multiFuture_GaussianWorldModel_20261005_092950/epoch_0450.pt"
WM_BACKEND = "multifuture"
WM_PLANNING_MODE = "expected_independent"
REWARD = esd.margin_tiebreak_reward()

# ---- ES / sim (= slurm_es_fail1500_wm_mf.sh) --------------------------------
ES = dict(population_size=64, elite_size=8, num_iterations=4, resample_timesteps=5)
START_TIMESTEP = 10
REPLAN_INTERVAL = 10
SIM_AGENTS = "log"     # use "idm" for the reactivity cases
SEED = 0

### Candidates

Clean scenarios (not offroad/colliding at t=10, goal > 20 m away) that **fail in all four ES runs** (latest.pt binary / tiebreak, e450 tiebreak idm / log), i.e. not seed noise. Labels are the e450 tiebreak log outcome.

In [ ]:
CANDIDATES = {
    # collision in every ES run (47 total); 00009#388 / 00017#173 still reach the goal -> clean dodge tests
    "collision": [("00009", 388), ("00017", 173), ("00001", 243), ("00013", 133), ("00023", 131)],
    # offroad in every ES run although the logged ego never leaves the road (21 total)
    "offroad":   [("00011", 168), ("00091", 145), ("00092", 26), ("00106", 375), ("00124", 425)],
    # goal missed in every ES run (163 total)
    "goal_miss": [("00002", 0), ("00002", 70), ("00003", 331), ("00006", 57)],
    # collide only with IDM agents -> run with SIM_AGENTS="idm"
    "reactive":  [("00101", 237), ("00152", 363)],
}

def tfrecord_path(shard: str) -> str:
    return f"{TFRECORD_DIR}/training_tfexample.tfrecord-{shard}-of-01000"

## 2. Load models

In [ ]:
policy, preprocess_cfg = esd.load_diffusion_policy(DIFFUSION_CKPT, seed=SEED)
wm = esd.load_world_model(WM_CKPT, backend=WM_BACKEND, planning_mode=WM_PLANNING_MODE)

baseline_planner = esd.DebugESPlanner(
    policy, preprocess_cfg,
    other_agent_forecast="wm", world_model_forecaster=wm,
    reward=REWARD, acquisition=esd.topk, initial_pool=esd.standard_pool, label="baseline",
    **ES,
)
# Only the initial pool changes; reward, acquisition and the ES loop stay the same.
variants = {
    "baseline": baseline_planner,
    "fps_x4": baseline_planner.variant(label="fps_x4", initial_pool=esd.oversample_fps(4)),
    "goal_mix_25": baseline_planner.variant(label="goal_mix_25", initial_pool=esd.goal_mix(0.25)),
    # Rotate 48 of the 64 samples about the ego by -30..30 deg (12 headings), then re-noise/denoise
    # them with resample_timesteps so they become plausible plans again.
    "heading_fan": baseline_planner.variant(label="heading_fan", initial_pool=esd.heading_fan(30.0, 12)),
    # Same fan crossed with slower variants (braking / yielding).
    "heading_speed_fan": baseline_planner.variant(
        label="heading_speed_fan", initial_pool=esd.heading_fan(30.0, 8, speed_scales=(1.0, 0.6, 0.3))),
    # Curvature fan: arc / lane_change / swerve (out-and-back) / reverse (out, then the opposite way),
    # both sides, ±15/30°, over half or the whole horizon, + smooth braking to a stop (34 shapes).
    # Fan is denoised at noise_scale=1.0 (the schedule's level); the ES iterations stay production (2.0).
    "shape_fan": baseline_planner.variant(label="shape_fan", initial_pool=esd.shape_fan()),
    # Same fan, and the ES re-noise step also at noise_scale=1.0 (less jitter in iterations 1-4).
    "shape_fan_es1": baseline_planner.variant(label="shape_fan_es1", initial_pool=esd.shape_fan(), resample_noise_scale=1.0),
    # + collision tiebreak: among plans that all collide, later contact ranks higher (not in production).
    "shape_fan_ctb": baseline_planner.variant(label="shape_fan_ctb", initial_pool=esd.shape_fan(),
        resample_noise_scale=1.0, reward=esd.margin_tiebreak_reward(collision_eps=1e-2)),
    # Tiebreak alone (production pool): does the signal help without the fan?
    "ctb_only": baseline_planner.variant(label="ctb_only", reward=esd.margin_tiebreak_reward(collision_eps=1e-2)),
    # Same profiles at ±20° and two speeds.
    "shape_speed_fan": baseline_planner.variant(label="shape_speed_fan", initial_pool=esd.shape_fan(
        amplitudes_deg=(20.0,), speed_scales=(1.0, 0.5))),
}

## 3. One scenario: was there a safe plan in the first pool?

Runs the baseline closed loop, finds the first failure, and re-plans with every variant at the replan steps leading up to it (same state, same seed). `iter 0` = initial pool, `iter 4` = after ES.

In [ ]:
CATEGORY, (SHARD, SCENARIO_IDX) = "collision", CANDIDATES["collision"][0]   # <- pick one

scenario = esd.load_scenario(tfrecord_path(SHARD), SCENARIO_IDX, lane_graph_dir=LANE_GRAPH_DIR)
base = esd.run_closed_loop(
    baseline_planner, scenario,
    start_timestep=START_TIMESTEP, replan_interval_steps=REPLAN_INTERVAL,
    sim_agents=SIM_AGENTS, seed=SEED, verbose=False,
)
print(base.report())
t_fail = esd.first_failure_t(base)
print("first failure t:", t_fail)

In [ ]:
esd.plot_timeline(base);

In [ ]:
# Decisions that could still have avoided the failure: the plan executing at t_fail and the two before it.
fail_anchor = base.anchor_for(t_fail if t_fail is not None else base.replan_steps[-1])
ANCHORS = [s for s in base.replan_steps if fail_anchor - 2 * REPLAN_INTERVAL <= s <= fail_anchor]

rows, traces = [], {}
for anchor in ANCHORS:
    for name, planner in variants.items():
        tr = base.traces[anchor] if name == "baseline" else esd.replan_at(planner, base, anchor, seed=SEED)
        traces[(anchor, name)] = tr
        for it in (0, -1):
            rows.append({"anchor": anchor, "variant": name, **esd.pool_stats(tr, it)})
pool_df = pd.DataFrame(rows).set_index(["anchor", "variant", "iter"])
pool_df[["n_safe", "frac_safe", "frac_collision_free", "frac_onroad", "best_score",
         "endpoint_pairwise_m", "endpoint_std_across_m", "endpoint_std_along_m", "final_speed_std_mps"]].round(3)

In [ ]:
# Look at one decision: candidates per ES iteration (bold = elites, red = chosen plan).
A, V = ANCHORS[-1], "fps_x4"   # <- anchor, variant
esd.plot_es_iterations(traces[(A, V)], base.snapshots[A], ego_idx=scenario.ego_idx, view_m=35);

### What the fans look like

`shape_fan` changes each sample's *curvature*: `arc` keeps turning, `lane_change` goes out and straightens, `swerve` goes out and comes back, `reverse` goes out then curves the opposite way; both sides, plus straight plans braking smoothly to a stop. The fan is then re-noised and denoised (SDEdit).

**Jitter:** the policy's `resample_from_condition` defaults to `noise_scale=2.0` (twice the schedule's noise), which 5 denoising steps cannot fully remove; it shows as zigzags on slow plans. The cell below shows only the initial pool (`num_iterations=0`) for raw / σ=2 / σ=1 / σ=1 with 10 steps.

In [ ]:
A = ANCHORS[-1]
# Initial pool only (num_iterations=0): is the denoised fan itself smooth? Compare noise scales / steps.
fan_only = lambda label, pool: baseline_planner.variant(label=label, initial_pool=pool, num_iterations=0)
fan_traces = {
    "raw": esd.replan_at(fan_only("fan raw", esd.shape_fan(denoise=False)), base, A, seed=SEED),
    "σ=2 (production)": esd.replan_at(fan_only("fan σ2", esd.shape_fan(noise_scale=2.0)), base, A, seed=SEED),
    "σ=1": esd.replan_at(fan_only("fan σ1", esd.shape_fan(noise_scale=1.0)), base, A, seed=SEED),
    "σ=1, 10 steps": esd.replan_at(fan_only("fan σ1 10st", esd.shape_fan(noise_scale=1.0, denoise_steps=10)), base, A, seed=SEED),
}
for name, tr in fan_traces.items():
    print(name, {k: round(v, 2) for k, v in esd.pool_stats(tr, 0).items()
                 if k in ("n_safe", "endpoint_std_across_m", "heading_change_std_deg", "best_score")})
    esd.plot_es_iterations(tr, base.snapshots[A], ego_idx=scenario.ego_idx, view_m=35);

### Does ES have a signal?

If every plan collides, scores tie and `top_k` picks elites by index (the unmodified samples at the front), so ES collapses back to the straight plan. `margin_tiebreak_reward(collision_eps=...)` adds `collision_eps × collision_time_tiebreak`: the mode-probability-weighted first-contact time against the WM futures (1 = no contact). `1e-2` sits above the offroad tiebreak (`1e-3`) and far below a collision-free plan (≥ 0.5). Below: distinct scores per iteration and the ES run for the fan without / with the tiebreak, and the tiebreak alone.

In [ ]:
for name in ("shape_fan_es1", "shape_fan_ctb", "ctb_only"):
    tr = esd.replan_at(variants[name], base, A, seed=SEED)
    print(name, "distinct scores per iter:", [np.unique(np.round(it.scores, 6)).size for it in tr.iterations],
          "best:", round(tr.best_score, 4))
    esd.plot_es_iterations(tr, base.snapshots[A], ego_idx=scenario.ego_idx, view_m=35);

In [ ]:
# Closed loop from scratch with each variant (same seed).
runs = {"baseline": base}
for name, planner in variants.items():
    if name != "baseline":
        runs[name] = esd.run_closed_loop(
            planner, scenario,
            start_timestep=START_TIMESTEP, replan_interval_steps=REPLAN_INTERVAL,
            sim_agents=SIM_AGENTS, seed=SEED, verbose=False,
        )
        print(runs[name].report())
esd.plot_timeline(base, compare=[r for k, r in runs.items() if k != "baseline"]);

In [ ]:
V = "fps_x4"
esd.animate_closed_loop(runs[V], view_m=40, show_plans=True, show_elites=True)

## 4. Sweep over the candidates

For each scenario: baseline closed loop → re-plan with every variant at the decisions before the failure (pool stats) → closed loop with every variant. Each scenario is a few minutes per variant; trim `CANDIDATES` first if needed. Reactive cases use IDM agents.

In [ ]:
def run_case(category, shard, idx, seed=SEED):
    sim_agents = "idm" if category == "reactive" else SIM_AGENTS
    sc = esd.load_scenario(tfrecord_path(shard), idx, lane_graph_dir=LANE_GRAPH_DIR)
    loop_kw = dict(start_timestep=START_TIMESTEP, replan_interval_steps=REPLAN_INTERVAL,
                   sim_agents=sim_agents, seed=seed, verbose=False)
    b = esd.run_closed_loop(baseline_planner, sc, **loop_kw)
    t_fail = esd.first_failure_t(b)
    fa = b.anchor_for(t_fail if t_fail is not None else b.replan_steps[-1])
    anchors = [s for s in b.replan_steps if fa - 2 * REPLAN_INTERVAL <= s <= fa]
    pool_rows, loop_rows = [], []
    for name, planner in variants.items():
        for a in anchors:
            tr = b.traces[a] if name == "baseline" else esd.replan_at(planner, b, a, seed=seed)
            p0, pl = esd.pool_stats(tr, 0), esd.pool_stats(tr, -1)
            pool_rows.append({"category": category, "scenario": f"{shard}#{idx}", "variant": name, "anchor": a,
                              "n_safe_init": p0["n_safe"], "n_safe_final": pl["n_safe"],
                              "spread_init_m": p0["endpoint_pairwise_m"], "across_init_m": p0["endpoint_std_across_m"]})
        r = b if name == "baseline" else esd.run_closed_loop(planner, sc, **loop_kw)
        loop_rows.append({"category": category, "scenario": f"{shard}#{idx}", "variant": name,
                          "success": r.success, "overlap": r.overlap, "offroad": r.offroad,
                          "goal_reached": r.goal_reached, "min_goal_dist_m": round(r.min_goal_distance_m, 1),
                          "first_fail_t": esd.first_failure_t(r)})
    return pool_rows, loop_rows

POOL, LOOP = [], []
for category, cases in CANDIDATES.items():
    for shard, idx in cases:
        print(f"--- {category} {shard}#{idx}", flush=True)
        try:
            p, l = run_case(category, shard, idx)
        except Exception as e:  # keep the sweep going
            print("   failed:", repr(e))
            continue
        POOL += p; LOOP += l
        print("   " + "  ".join(f"{r['variant']}={'OK' if r['success'] else 'fail'}" for r in l), flush=True)

In [ ]:
# Diagnosis: did the *baseline* initial pool contain a safe plan at the decisions before the failure?
pool = pd.DataFrame(POOL)
diag = (pool[pool.variant == "baseline"]
        .groupby(["category", "scenario"])
        .agg(min_safe_init=("n_safe_init", "min"), min_safe_final=("n_safe_final", "min"),
             spread_init_m=("spread_init_m", "mean")))
diag["verdict"] = np.where(diag.min_safe_init == 0, "no safe plan sampled (diversity)",
                  np.where(diag.min_safe_final == 0, "safe plan lost by ES", "safe plan available (scoring/WM)"))
display(diag)
display(diag.verdict.value_counts())

In [ ]:
# Same decisions, per variant: safe plans in the initial pool and its spread.
display(pool.pivot_table(index=["category", "scenario", "anchor"], columns="variant",
                         values=["n_safe_init", "across_init_m"]).round(1))

In [ ]:
# Closed-loop outcome per variant.
loop = pd.DataFrame(LOOP)
display(loop.pivot_table(index=["category", "scenario"], columns="variant", values="success", aggfunc="first"))
display(loop.groupby("variant")[["success", "overlap", "offroad", "goal_reached"]].mean().round(2))

**Reading the result**

* Baseline is all failures by construction, so any success in a variant column is a recovery. Re-run with 2–3 seeds before trusting a single flip: the same config can flip on a different seed (00022#17 did).
* If most `verdict`s are *safe plan available*, the next thing to fix is scoring/the WM forecast, not diversity.
* For "more reactive", also try `REPLAN_INTERVAL = 5`; that changes how often the ego reacts, separately from what it can propose.